In [15]:
%pip install openai python-dotenv

  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/56/73/c605f0cee1882e453a4b7fc2f17f4a42a7d4c1ed2304aff4b163ffade819/openai-3.37.0-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/44.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/44.5 kB ? eta -:--:--
     ----------------- -------------------- 20.5/44.5 kB 162.5 kB/s eta 0:00:01
     -------------------------- ----------- 30.7/44.5 kB 186.2 kB/s eta 0:00:01
     -------------------------------------- 44.5/44.5 kB 243.3 kB/s eta 0:00:00
  Obtaining dependency information for python-dotenv from https://files.pythonhosted.org/packages/60/d1/38f3a3405989a89ac18390803e70c6ad7c7760da4f9b83cbeca0c44a0c72/python_dotenv-1.2.4-py3-none-any.whl.metadata
  Obtaining dependency information for jiter<1,>=0.16.0 from https://files.pythonhosted.org/packages/4d/3e/f17099d841ccb1603d1223941b845bf38503f540aa6be8e9d50c8488f98f/jiter-0.18.0-cp311-cp311-win_amd64.whl

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import sqlite3

gym_info = pd.read_csv("gym_info.csv")
membership_plans = pd.read_csv("membership_plans.csv")

print("=== Gym Information ===")
display(gym_info)

print("=== Membership Plans ===")
display(membership_plans)

=== Gym Information ===


,topic,content
0,opening_hours,平日營業時間 08:00–23:00
1,weekend_hours,週末營業時間 08:00–22:00
2,address,高雄市楠梓區大學路100號
3,phone,07-123-4567
4,parking,提供汽車及機車停車位
5,payment,支援現金、信用卡及行動支付
6,trial,新會員可預約一次免費體驗教練課程
7,personal_training,提供一對一私人教練課程
8,shower,提供淋浴間及置物櫃
9,cancellation,會員方案取消請洽櫃檯辦理


=== Membership Plans ===


,plan_name,category,price
0,單次入場,Entry,150
1,一週體驗,Entry,500
2,月費會員,Membership,899
3,季度會員,Membership,3300
4,一年會員,Membership,10500
5,二年會員,Membership,20000
6,學生月費方案,Student,799
7,學生季度方案,Student,3000
8,學生年度方案,Student,9500
9,單堂私人教練,Personal Training,1200


In [2]:
# 建立 SQLite 資料庫
conn = sqlite3.connect("gym.db")

# 將兩個 DataFrame 寫入資料庫
gym_info.to_sql(
    "gym_info",
    conn,
    if_exists="replace",
    index=False
)

membership_plans.to_sql(
    "membership_plans",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("gym.db created successfully!")

gym.db created successfully!


In [3]:
# 連接資料庫
conn = sqlite3.connect("gym.db")

# 查詢 gym_info
gym_test = pd.read_sql_query(
    "SELECT * FROM gym_info",
    conn
)

# 查詢 membership_plans
plan_test = pd.read_sql_query(
    "SELECT * FROM membership_plans",
    conn
)

conn.close()

print("=== gym_info table ===")
display(gym_test)

print("=== membership_plans table ===")
display(plan_test)

=== gym_info table ===


,topic,content
0,opening_hours,平日營業時間 08:00–23:00
1,weekend_hours,週末營業時間 08:00–22:00
2,address,高雄市楠梓區大學路100號
3,phone,07-123-4567
4,parking,提供汽車及機車停車位
5,payment,支援現金、信用卡及行動支付
6,trial,新會員可預約一次免費體驗教練課程
7,personal_training,提供一對一私人教練課程
8,shower,提供淋浴間及置物櫃
9,cancellation,會員方案取消請洽櫃檯辦理


=== membership_plans table ===


,plan_name,category,price
0,單次入場,Entry,150
1,一週體驗,Entry,500
2,月費會員,Membership,899
3,季度會員,Membership,3300
4,一年會員,Membership,10500
5,二年會員,Membership,20000
6,學生月費方案,Student,799
7,學生季度方案,Student,3000
8,學生年度方案,Student,9500
9,單堂私人教練,Personal Training,1200


In [6]:
def get_gym_info(topic):
    conn = sqlite3.connect("gym.db")

    cursor = conn.cursor()
    cursor.execute(
        "SELECT content FROM gym_info WHERE topic = ?",
        (topic,)
    )

    result = cursor.fetchone()
    conn.close()

    if result:
        return result[0]
    else:
        return "找不到相關的健身房資訊"

In [7]:
print(get_gym_info("opening_hours"))
print(get_gym_info("parking"))
print(get_gym_info("shower"))

平日營業時間 08:00–23:00
提供汽車及機車停車位
提供淋浴間及置物櫃


In [8]:
def get_membership_plan(plan_name):
    conn = sqlite3.connect("gym.db")

    cursor = conn.cursor()
    cursor.execute(
        "SELECT plan_name, category, price FROM membership_plans WHERE plan_name = ?",
        (plan_name,)
    )

    result = cursor.fetchone()
    conn.close()

    if result:
        return {
            "plan_name": result[0],
            "category": result[1],
            "price": result[2]
        }
    else:
        return "找不到這個方案"

In [9]:
print(get_membership_plan("月費會員"))
print(get_membership_plan("學生月費方案"))
print(get_membership_plan("單堂私人教練"))

{'plan_name': '月費會員', 'category': 'Membership', 'price': 899}
{'plan_name': '學生月費方案', 'category': 'Student', 'price': 799}
{'plan_name': '單堂私人教練', 'category': 'Personal Training', 'price': 1200}


In [10]:
def calculate_membership_fee(plan_name, quantity=1):
    conn = sqlite3.connect("gym.db")

    cursor = conn.cursor()
    cursor.execute(
        "SELECT price FROM membership_plans WHERE plan_name = ?",
        (plan_name,)
    )

    result = cursor.fetchone()
    conn.close()

    if not result:
        return "找不到這個方案"

    price = result[0]
    total = price * quantity

    return {
        "plan_name": plan_name,
        "unit_price": price,
        "quantity": quantity,
        "total": total
    }

In [11]:
print(calculate_membership_fee("月費會員", 2))
print(calculate_membership_fee("單堂私人教練", 3))
print(calculate_membership_fee("學生月費方案", 2))

{'plan_name': '月費會員', 'unit_price': 899, 'quantity': 2, 'total': 1798}
{'plan_name': '單堂私人教練', 'unit_price': 1200, 'quantity': 3, 'total': 3600}
{'plan_name': '學生月費方案', 'unit_price': 799, 'quantity': 2, 'total': 1598}


In [12]:
def transfer_to_staff(reason):
    return {
        "status": "transferred",
        "reason": reason,
        "message": "此問題需要由真人客服協助，已為您轉接健身房工作人員。"
    }

In [13]:
print(transfer_to_staff("會員要求申訴私人教練的服務態度"))

{'status': 'transferred', 'reason': '會員要求申訴私人教練的服務態度', 'message': '此問題需要由真人客服協助，已為您轉接健身房工作人員。'}


In [20]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("LLM_API_URL")
)

ai_model = os.getenv("MODEL_NAME")

response = client.chat.completions.create(
    model=ai_model,
    messages=[
        {"role": "user", "content": "你好，請簡單介紹你自己。"}
    ]
)

print(response.choices[0].message.content)



您好！我是由國立高雄科技大學提供的 AI 助理服務，很高興為您解答問題或提供協助。如果有任何需求，請隨時告訴我！


In [22]:
tools = [
    # 1. 查詢健身房基本資訊
    {
        "type": "function",
        "function": {
            "name": "get_gym_info",
            "description": "查詢健身房的基本資訊，例如營業時間、地址、電話、停車、付款方式、體驗課、私人教練、淋浴間與取消規定。",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "要查詢的資訊主題",
                        "enum": [
                            "opening_hours",
                            "weekend_hours",
                            "address",
                            "phone",
                            "parking",
                            "payment",
                            "trial",
                            "personal_training",
                            "shower",
                            "cancellation"
                        ]
                    }
                },
                "required": ["topic"]
            }
        }
    },

    # 2. 查詢會員方案
    {
        "type": "function",
        "function": {
            "name": "get_membership_plan",
            "description": "查詢健身房會員方案、入場方案、學生方案、私人教練或團體課程的價格。",
            "parameters": {
                "type": "object",
                "properties": {
                    "plan_name": {
                        "type": "string",
                        "description": "要查詢的方案名稱，例如月費會員、學生月費方案、單堂私人教練。"
                    }
                },
                "required": ["plan_name"]
            }
        }
    },

    # 3. 計算方案費用
    {
        "type": "function",
        "function": {
            "name": "calculate_membership_fee",
            "description": "根據方案名稱與數量計算總費用，例如計算兩個月會員費或三堂私人教練課的總價。",
            "parameters": {
                "type": "object",
                "properties": {
                    "plan_name": {
                        "type": "string",
                        "description": "方案名稱"
                    },
                    "quantity": {
                        "type": "integer",
                        "description": "購買數量或期數",
                        "minimum": 1
                    }
                },
                "required": ["plan_name", "quantity"]
            }
        }
    },

    # 4. 轉接真人客服
    {
        "type": "function",
        "function": {
            "name": "transfer_to_staff",
            "description": "當使用者需要申訴、退款爭議、特殊協助或其他需要真人處理的問題時，轉接健身房工作人員。",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {
                        "type": "string",
                        "description": "需要轉接真人客服的原因"
                    }
                },
                "required": ["reason"]
            }
        }
    }
]

print("Tools:", [tool["function"]["name"] for tool in tools])

Tools: ['get_gym_info', 'get_membership_plan', 'calculate_membership_fee', 'transfer_to_staff']


In [23]:
test_messages = [
    {
        "role": "user",
        "content": "請問你們健身房有停車位嗎？"
    }
]

response = client.chat.completions.create(
    model=ai_model,
    messages=test_messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

print("一般回答：", message.content)

if message.tool_calls:
    for call in message.tool_calls:
        print("選擇的 Tool：", call.function.name)
        print("參數：", call.function.arguments)
else:
    print("沒有呼叫 Tool")

一般回答： None
選擇的 Tool： get_gym_info
參數： {"topic": "parking"}


In [24]:
tool_functions = {
    "get_gym_info": get_gym_info,
    "get_membership_plan": get_membership_plan,
    "calculate_membership_fee": calculate_membership_fee,
    "transfer_to_staff": transfer_to_staff
}

In [25]:
print(tool_functions.keys())

dict_keys(['get_gym_info', 'get_membership_plan', 'calculate_membership_fee', 'transfer_to_staff'])


In [26]:
# 取得模型要求的第一個 Tool Call
tool_call = message.tool_calls[0]

tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)

print("Tool 名稱：", tool_name)
print("Tool 參數：", tool_args)

# 找到真正的 Python function 並執行
tool_result = tool_functions[tool_name](**tool_args)

print("Tool 執行結果：", tool_result)

Tool 名稱： get_gym_info
Tool 參數： {'topic': 'parking'}
Tool 執行結果： 提供汽車及機車停車位


In [27]:
# 把模型原本的 Tool Call 加進對話
test_messages.append(message)

# 把 Tool 執行結果交回模型
test_messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(tool_result)
})

# 再次呼叫 LLM，讓它根據 Tool 結果回答使用者
final_response = client.chat.completions.create(
    model=ai_model,
    messages=test_messages,
    tools=tools
)

final_answer = final_response.choices[0].message.content

print("最終回答：")
print(final_answer)

最終回答：


我們健身房提供汽車與機車停車位。若您需要更詳細的資訊（例如停車費用、車位數量或使用規定），請隨時告訴我！


In [74]:
def run_agent(user_message, max_steps=5):
    messages = [
        {
            "role": "system",
            "content": """
你是健身房的 AI 客服助理。

請根據使用者的問題，選擇適合的工具取得正確資訊。

規則：
1. 健身房基本資訊必須使用 get_gym_info 查詢。
2. 如果使用者詢問任何會員方案、入場、體驗方案、私人教練或課程的「單一價格」，必須使用 get_membership_plan；即使 gym_info 中有相關主題，只要問題是在詢問價格，就優先使用 get_membership_plan。
3. 如果使用者詢問多個、數量、次數、月份、堂數的總價或總費用，只要需要「單價 × 數量」的計算，必須優先使用 calculate_membership_fee，不要使用 get_membership_plan。
4. 遇到申訴、退款爭議、特殊問題或需要真人處理的情況，使用 transfer_to_staff。
5. 不要自行編造健身房資訊或價格。
6. 使用工具取得結果後，再用自然、簡潔的繁體中文回答使用者。
7. 只回答與本健身房服務相關的問題；如果使用者詢問與健身房無關的內容，請禮貌說明你只能協助處理健身房相關問題。
"""
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    tools_used = []

    for step in range(max_steps):

        response = client.chat.completions.create(
            model=ai_model,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message

        # 如果模型沒有呼叫 Tool，代表已經得到最終答案
        if not assistant_message.tool_calls:
            return {
                "answer": assistant_message.content,
                "tools_used": tools_used
            }

        # 把模型的 Tool Call 加進對話
        messages.append(assistant_message)

        # 執行模型要求的所有 Tools
        for tool_call in assistant_message.tool_calls:

            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            if tool_name not in tool_functions:
                tool_result = "錯誤：不存在這個工具"
            else:
                try:
                    tool_result = tool_functions[tool_name](**tool_args)
                except Exception as e:
                    tool_result = f"工具執行錯誤：{str(e)}"

            tools_used.append({
                "tool": tool_name,
                "arguments": tool_args,
                "result": tool_result
            })

            # 把 Tool 結果交回 LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(
                    tool_result,
                    ensure_ascii=False
                )
            })

    return {
        "answer": "抱歉，目前無法完成這個問題，請稍後再試。",
        "tools_used": tools_used
    }

In [62]:
test_questions = [
    "月費會員多少錢？",
    "如果我要買三堂單堂私人教練，總共多少錢？",
    "我要申訴私人教練的服務態度。"
]

for question in test_questions:
    print("=" * 60)
    print("問題：", question)

    result = run_agent(question)

    print("回答：", result["answer"])
    print("Tools used：", result["tools_used"])
    print()

問題： 月費會員多少錢？
回答： 
月費會員的費用為 **899 元**（每月）。若需要查詢其他方案或計算多個月的總費用，請告訴我！
Tools used： [{'tool': 'get_membership_plan', 'arguments': {'plan_name': '月費會員'}, 'result': {'plan_name': '月費會員', 'category': 'Membership', 'price': 899}}]

問題： 如果我要買三堂單堂私人教練，總共多少錢？
回答： 
三堂單堂私人教練的總費用為 **3,600 元**。
Tools used： [{'tool': 'calculate_membership_fee', 'arguments': {'plan_name': '單堂私人教練', 'quantity': 3}, 'result': {'plan_name': '單堂私人教練', 'unit_price': 1200, 'quantity': 3, 'total': 3600}}]

問題： 我要申訴私人教練的服務態度。
回答： 

已為您轉接真人客服協助。健身房工作人員將會接手您的申訴，並提供後續處理。謝謝！
Tools used： [{'tool': 'transfer_to_staff', 'arguments': {'reason': '申訴私人教練的服務態度'}, 'result': {'status': 'transferred', 'reason': '申訴私人教練的服務態度', 'message': '此問題需要由真人客服協助，已為您轉接健身房工作人員。'}}]



In [63]:
import gradio as gr

print("Gradio version:", gr.__version__)

Gradio version: 6.30.0


In [64]:
def chat_with_agent(message, history):
    result = run_agent(message)

    answer = result["answer"]

    return answer

In [65]:
demo = gr.ChatInterface(
    fn=chat_with_agent,
    title="🏋️ 健身房 AI 客服",
    description="歡迎使用健身房 AI 客服！可以詢問會員方案、價格、營業時間、停車資訊等問題。",
    examples=[
        "請問你們平日營業到幾點？",
        "月費會員多少錢？",
        "三堂單堂私人教練總共多少錢？",
        "你們有停車位嗎？",
        "我要申訴私人教練的服務態度。"
    ]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [66]:
demo.close()

Closing server running on port: 7864


In [67]:
def chat_with_tools(message, history):
    result = run_agent(message)

    answer = result["answer"]
    tools_used = result["tools_used"]

    # 整理 Tool 使用紀錄
    if tools_used:
        tool_text = ""

        for i, item in enumerate(tools_used, 1):
            tool_text += f"### Tool {i}: `{item['tool']}`\n"
            tool_text += f"- **Arguments:** `{item['arguments']}`\n"
            tool_text += f"- **Result:** `{item['result']}`\n\n"
    else:
        tool_text = "這次沒有使用 Tool。"

    return answer, tool_text

In [68]:
with gr.Blocks() as demo_v2:

    gr.Markdown(
        """
        # 🏋️ 健身房 AI 客服
        可以詢問健身房資訊、會員方案、價格，
        也可以協助計算費用或轉接真人客服。
        """
    )

    chatbot = gr.Chatbot(
        label="客服對話",
        height=400
    )

    with gr.Row():
        message_box = gr.Textbox(
            label="輸入問題",
            placeholder="例如：月費會員多少錢？",
            scale=5
        )

        send_button = gr.Button(
            "送出",
            variant="primary",
            scale=1
        )

    gr.Markdown("## 🔧 Tools Used")

    tool_output = gr.Markdown(
        "尚未使用任何 Tool。"
    )

    def respond(message, history):

        answer, tool_text = chat_with_tools(
            message,
            history
        )

        history = history + [
            {
                "role": "user",
                "content": message
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]

        return "", history, tool_text

    send_button.click(
        fn=respond,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot, tool_output]
    )

    message_box.submit(
        fn=respond,
        inputs=[message_box, chatbot],
        outputs=[message_box, chatbot, tool_output]
    )

demo_v2.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [69]:
tool_choice_tests = [
    # get_gym_info
    ("請問你們平日幾點營業？", "get_gym_info"),
    ("健身房有停車位嗎？", "get_gym_info"),
    ("請問可以刷信用卡嗎？", "get_gym_info"),

    # get_membership_plan
    ("月費會員多少錢？", "get_membership_plan"),
    ("學生月費方案的價格是多少？", "get_membership_plan"),
    ("一週體驗多少錢？", "get_membership_plan"),

    # calculate_membership_fee
    ("如果我要買三堂單堂私人教練，總共多少錢？", "calculate_membership_fee"),
    ("兩個月的月費會員總共多少錢？", "calculate_membership_fee"),
    ("我要買四次單次入場，總共多少？", "calculate_membership_fee"),

    # transfer_to_staff
    ("我要申訴私人教練的服務態度。", "transfer_to_staff"),
    ("我要反映工作人員服務態度不好。", "transfer_to_staff"),
    ("我的會員扣款有爭議，我要找真人處理。", "transfer_to_staff"),
]

In [70]:
def run_tool_choice_test(test_cases):
    correct = 0
    results = []

    for i, (question, expected_tool) in enumerate(test_cases, 1):

        result = run_agent(question)

        if result["tools_used"]:
            actual_tool = result["tools_used"][0]["tool"]
        else:
            actual_tool = "None"

        is_correct = actual_tool == expected_tool

        if is_correct:
            correct += 1

        results.append({
            "No.": i,
            "Question": question,
            "Expected": expected_tool,
            "Actual": actual_tool,
            "Correct": "✓" if is_correct else "✗"
        })

    score = correct / len(test_cases) * 100

    return results, correct, score

In [75]:
import pandas as pd

results, correct, score = run_tool_choice_test(tool_choice_tests)

df_results = pd.DataFrame(results)

display(df_results)

print("=" * 60)
print(f"Tool-Choice Test：{correct}/{len(tool_choice_tests)}")
print(f"Score：{score:.1f}%")

,No.,Question,Expected,Actual,Correct
0,1,請問你們平日幾點營業？,get_gym_info,get_gym_info,✓
1,2,健身房有停車位嗎？,get_gym_info,get_gym_info,✓
2,3,請問可以刷信用卡嗎？,get_gym_info,get_gym_info,✓
3,4,月費會員多少錢？,get_membership_plan,get_membership_plan,✓
4,5,學生月費方案的價格是多少？,get_membership_plan,get_membership_plan,✓
5,6,一週體驗多少錢？,get_membership_plan,get_membership_plan,✓
6,7,如果我要買三堂單堂私人教練，總共多少錢？,calculate_membership_fee,calculate_membership_fee,✓
7,8,兩個月的月費會員總共多少錢？,calculate_membership_fee,calculate_membership_fee,✓
8,9,我要買四次單次入場，總共多少？,calculate_membership_fee,get_membership_plan,✗
9,10,我要申訴私人教練的服務態度。,transfer_to_staff,transfer_to_staff,✓


Tool-Choice Test：11/12
Score：91.7%
